In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from diffusion_geometry.visualisation import *
from plotly.subplots import make_subplots
from figures.generate_data import load_image_point_cloud
from diffusion_geometry import DiffusionGeometry


In [2]:
camera = dict(eye=dict(x=0.8, y=-0.9, z=1.),
                center=dict(x=0, y=0, z=-0.2),
                up=dict(x=0, y=0, z=1))
# rescale eye:
camera['eye'] = {k: v * 1.3 for k, v in camera['eye'].items()}

## Connection Laplacian

The Levi-Civita connection defines an operator $\nabla: \mathfrak{X}(M) \to \Omega^{0,2}(M)$ from a vector field $X$ to its derivative $\nabla X$, which is a $(0,2)$-tensor. 

The composition of this map with its adjoint defines an operator $\nabla^*\nabla: \mathfrak{X}(M) \to \mathfrak{X}(M)$ called the **connection Laplacian** (or the 'Bochner' or 'rough' Laplacian). Like the Hodge Laplacian, we can diagonalise it to find 'smooth' vector fields. This is the problem addressed by 'Vector Diffusion Maps' (Singer and Wu, 2012). We can straightforwardly implement it with diffusion geometry.

In [66]:
from diffusion_geometry import DiffusionGeometry
from figures.generate_data import gen_3d_data

# Generate torus data:
n = 2000
data = gen_3d_data(kind='torus', minor_radius=1.0, major_radius=2.0, n=n, noise=0.0, seed=0)[0]

# Vector diffusion maps:
dg = DiffusionGeometry.from_point_cloud(data)
connection = dg.levi_civita.adjoint @ dg.levi_civita
eigenvalues, eigenvectors = connection.spectrum()

# Plot the first 3 eigenvectors in a single row:
fig = make_subplots(
    rows=1,
    cols=3,
    specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
    horizontal_spacing=0.02,
)

for k in range(3):
    fig_k = plot_quiver_3d(data, eigenvectors[k].to_ambient(), scale=0.1, camera=camera, arrow_scale=0.6)
    fig.add_traces(list(fig_k.data), rows=[1] * len(fig_k.data), cols=[k + 1] * len(fig_k.data))

scene_layout = dict(camera=camera, xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False))
fig.update_layout(
    scene=scene_layout,
    scene2=scene_layout,
    scene3=scene_layout,
    margin=dict(l=0, r=0, t=0, b=0),
)

fig.show()

In [4]:
from diffusion_geometry.operators import LinearOperator

sharp = LinearOperator(dg.form_space(1), dg.vector_field_space, weak_matrix=np.eye(dg.form_space(1).dim))
flat = LinearOperator(dg.vector_field_space, dg.form_space(1), weak_matrix=np.eye(dg.vector_field_space.dim))

ricci = sharp @ dg.laplacian(1) @ flat - connection

a = eigenvectors[0]

dg.g(a, ricci(a))
plot_scatter_3d(data, color=dg.g(a, ricci(a)), camera=camera).show()

In [ ]:
def ricci(a):
    dg = a.dg
    term1 = (dg.function(a.pointwise_norm()**2).laplacian() / 2).to_ambient()
    term2 = - dg.levi_civita(a).pointwise_norm()**2
    term3 = dg.g(a.flat(), dg.laplacian(1)(a.flat()))
    return term1 + term2 + term3

vals, vecs = dg.laplacian(1).spectrum()
a = vecs[0].sharp()
b = vecs[1].sharp()
plot_scatter_3d(data, color=ricci(a), camera=camera).show()
plot_scatter_3d(data, color=dg.sectional_curvature(a,b), camera=camera).show()

In [ ]:
def batched_generalized_eigh(A, B):
    """
    Solves the generalized eigenvalue problem Av = λBv for a batch of matrices.
    
    Args:
        A: (Batch, N, N) Symmetric matrix batch
        B: (Batch, N, N) Symmetric, Positive-Definite matrix batch
        
    Returns:
        vals: (Batch, N) Eigenvalues
        vecs: (Batch, N, N) Eigenvectors (column-wise)
    """
    # 1. Cholesky decomposition: B = L @ L.T
    try:
        L = np.linalg.cholesky(B)
    except np.linalg.LinAlgError:
        raise ValueError("Matrix B must be positive definite for Cholesky decomposition.")

    # 2. Transform A to standard form: C = L^-1 @ A @ L^-T
    # We use solve(L, A) which computes L^-1 @ A
    # Then we swapaxes to handle the L^-T part efficiently
    Y = np.linalg.solve(L, A)
    C = np.linalg.solve(L, np.swapaxes(Y, -1, -2))
    C = np.swapaxes(C, -1, -2) # Make symmetric again for eigh

    # 3. Solve standard eigenvalue problem: C @ y = λ @ y
    vals, y = np.linalg.eigh(C)

    # 4. Transform eigenvectors back: v = L^-T @ y
    # This corresponds to solving L.T @ v = y
    # We transpose L manually for the solve
    vecs = np.linalg.solve(np.swapaxes(L, -1, -2), y)

    return vals, vecs

In [153]:
dg.immersion_coords.T.shape

(3, 2000)

In [208]:
from diffusion_geometry import DiffusionGeometry
from figures.generate_data import gen_3d_data

# Generate torus data:
n = 2000
data = gen_3d_data(kind='torus', minor_radius=1.0, major_radius=2.0, n=n, noise=0., seed=0)[0]
dg = DiffusionGeometry.from_point_cloud(data, n0 = 1000)

x = dg.function(dg.immersion_coords.T)
gamma = dg.cache.gamma_coords_regularised

term1 = - dg.function(gamma.T).laplacian().to_ambient().T
term2 = dg.g(x[None,:].d(), x[:,None].laplacian().d()).T
# term2 = dg.triple.cdc(x.to_ambient().T, x.laplacian().to_ambient().T)
gamma2 = (term1 + term2 + term2.transpose((0,2,1))) / 2
# gamma2 = dg._regularise(gamma2)

vals, vecs = batched_generalized_eigh(gamma2, gamma)
# vals = dg._regularise(vals)

plot_scatter_3d(data, color=vals[:,1], 
                camera=camera, 
                # amax=0.7
                ).show()
# plot_scatter_3d(data, color=term2[:,0,0], camera=camera).show()

In [203]:
vals.mean(axis=0).round(1)

array([-0.5, -0. , 21.4])